# ✈️ FériasBot — Assistente de Viagens

**Como usar:**
1. Execute a célula abaixo clicando no botão ▶️ (ou Shift+Enter)
2. Cole sua chave da API Anthropic no campo que aparecer
3. Digite sua pergunta e clique em **Enviar**

> 🔑 Obtenha sua chave gratuita em: https://console.anthropic.com

In [ ]:
# Instala as dependências (só na primeira vez)
!pip install anthropic -q

import anthropic
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

SYSTEM_PROMPT = """Você é um agente especializado em férias e viagens, chamado FériasBot.
Responda sempre em português brasileiro, de forma entusiasmada e prática.
Ajude com destinos, roteiros, hospedagem, transporte, orçamento, documentação,
dicas culturais, segurança e atividades turísticas. Use emojis ocasionalmente."""

# ── Widgets ──────────────────────────────────────────────
api_key_input = widgets.Password(
    placeholder='Cole aqui sua chave (sk-ant-...)',
    description='🔑 API Key:',
    layout=widgets.Layout(width='500px')
)

user_input = widgets.Text(
    placeholder='Ex: Qual o melhor destino de praia no Brasil?',
    description='Você:',
    layout=widgets.Layout(width='500px')
)

send_btn = widgets.Button(
    description='Enviar ✈️',
    button_style='primary',
    layout=widgets.Layout(width='120px')
)

clear_btn = widgets.Button(
    description='Limpar 🗑️',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)

output_area = widgets.Output()

conversation_history = []

def render_message(role, text):
    if role == 'user':
        return f'<div style="background:#e3f2fd;border-radius:10px;padding:10px 14px;margin:6px 0;max-width:85%;margin-left:auto"><b>🧑 Você</b><br>{text}</div>'
    else:
        return f'<div style="background:#f1f8e9;border-radius:10px;padding:10px 14px;margin:6px 0;max-width:85%"><b>✈️ FériasBot</b><br>{text}</div>'

def redraw_chat():
    with output_area:
        clear_output(wait=True)
        html = '<div style="font-family:sans-serif;max-width:620px">'
        if not conversation_history:
            html += '<p style="color:#888">Olá! 🌴 Me pergunte qualquer coisa sobre viagens e férias!</p>'
        for msg in conversation_history:
            html += render_message(msg['role'], msg['content'].replace('\n', '<br>'))
        html += '</div>'
        display(HTML(html))

def on_send(btn):
    key = api_key_input.value.strip()
    question = user_input.value.strip()

    if not key:
        with output_area:
            clear_output(wait=True)
            display(HTML('<p style="color:red">⚠️ Cole sua chave da API no campo acima.</p>'))
        return

    if not question:
        return

    user_input.value = ''
    conversation_history.append({'role': 'user', 'content': question})

    with output_area:
        clear_output(wait=True)
        html = '<div style="font-family:sans-serif;max-width:620px">'
        for msg in conversation_history:
            html += render_message(msg['role'], msg['content'].replace('\n', '<br>'))
        html += render_message('assistant', '⏳ Pensando...')
        html += '</div>'
        display(HTML(html))

    try:
        client = anthropic.Anthropic(api_key=key)
        response = client.messages.create(
            model='claude-opus-4-7',
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=conversation_history,
        )
        answer = response.content[0].text
        conversation_history.append({'role': 'assistant', 'content': answer})
    except anthropic.AuthenticationError:
        conversation_history.pop()
        with output_area:
            clear_output(wait=True)
            display(HTML('<p style="color:red">❌ Chave inválida. Verifique em console.anthropic.com</p>'))
        redraw_chat()
        return
    except Exception as e:
        conversation_history.pop()
        with output_area:
            clear_output(wait=True)
            display(HTML(f'<p style="color:red">❌ Erro: {e}</p>'))
        redraw_chat()
        return

    redraw_chat()

def on_clear(btn):
    conversation_history.clear()
    redraw_chat()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
user_input.on_submit(on_send)

display(
    api_key_input,
    widgets.HBox([user_input, send_btn, clear_btn]),
    output_area
)
redraw_chat()